# Module 06 — Behavioral Cloning

Imitation learning — the dominant paradigm in modern robotics. See [the lesson](README.md).

> Tip: in Colab, set **Runtime → Change runtime type → GPU**

In [ ]:
# === Colab setup: run me first ===
import os, sys
if not os.path.exists('rl'):
    # On Colab, clone the repo so the `rl` package is importable.
    !git clone https://github.com/anhduckkzz/lunarlander.git repo && (cp -r repo/* . 2>/dev/null || true)
    !pip -q install 'gymnasium[box2d]>=0.29' torch numpy matplotlib imageio tqdm
import torch
print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## Notebook type and ordered study structure

**Notebook type:** Type A — core mechanism notebook. This should be studied slowly because the mechanism appears again and again across the whole repo.

**Your objective in this notebook:** Learn behavioral cloning as supervised robot policy learning.

Use this exact order every time:

1. **Why this matters** — identify the real problem this topic solves.
2. **Mental model** — explain the idea in plain language before symbols.
3. **Math mechanism** — write the smallest formula and define every symbol.
4. **From-scratch code** — run/read the simple implementation slowly.
5. **Line-by-line explanation** — trace inputs, internal variables, update rule, and outputs.
6. **Debug/visualize** — print shapes, values, curves, maps, or weights so behavior is visible.
7. **Framework version** — map the mechanism to a real library/API.
8. **Scratch → framework mapping** — write what the framework hides and what it exposes.
9. **Real-system role** — place the topic inside robotics, driving, drones, manipulation, or VLA.
10. **Failure modes** — list how it breaks and how you would notice.
11. **Exercises** — change parameters, break the example, and explain the result.
12. **Mini-project** — build a small artifact you can keep.
13. **Next step** — choose the next notebook or tool.

**Mental model for this topic:** If expert demonstrations are good, learning control can start as input-output prediction.

**Core math / mechanism to keep in mind:** min sum loss(pi(o_i), a_i_expert).

**Recommended debug habit:** after every code cell, ask “what are the inputs, what changed, and what would be unsafe or wrong in a real robot?”

## From-scratch focus and code-reading checklist

**Scratch focus:** Clone a DQN/PPO-style expert and evaluate covariate shift.

When reading code in this notebook or the matching repo module, trace it like this:

| Step | Question to answer |
|---|---|
| Input | What is the state, observation, tensor, point, reward, or measurement? |
| Representation | Is it a scalar, vector, matrix, image, point cloud, token sequence, or action? |
| Mechanism | Which line implements the math/update rule? |
| Parameters | Which numbers are hyperparameters, physical constants, or learned weights? |
| Output | What changed after the step? |
| Debug signal | What should I print/plot to know it is working? |

Do not move to the framework/API version until you can explain the scratch version without reading the code comments.

## Visualization and debugging ideas

Use at least one of these while studying:

- Print tensor/vector shapes before and after the core operation.
- Print the first few values before and after an update.
- Plot a curve when there is learning, control, filtering, planning, or optimization.
- Draw frames, maps, paths, sensor rays, or attention matrices when geometry is involved.
- Change one parameter at a time and predict the effect before running.

For this notebook, a useful first visualization/debug target is: **Clone a DQN/PPO-style expert and evaluate covariate shift.**

## Scratch → framework mapping

| From-scratch idea in this repo | Practical framework/API | What to learn from the framework |
|---|---|---|
| BehavioralCloning | LeRobot policy training | LeRobot packages datasets and policy architectures |
| collect_demonstrations | robot dataset recorder | real systems log observations/actions/timestamps |
| supervised loss | ACT/diffusion policy objectives | modern robot policies mostly start from imitation |

**Framework learning rule:** do not memorize the API first. First identify which scratch concept it replaces, then learn its inputs, outputs, configuration, and failure modes.

## Real-system application

Imitation learning is the entry point for robot/VLA policies because real rewards are hard and exploration is unsafe.

Ask these system questions:

1. What module produces the input to this component?
2. What module consumes its output?
3. What latency, safety, calibration, or data-format assumptions exist?
4. What metric tells me this component is good enough for the larger system?

## Failure modes and debugging

Common ways this topic can fail:

- covariate shift
- bad demonstrations
- action normalization mismatch
- missing recovery data

For each failure, write:

- **Symptom:** what would I see in logs, plots, robot behavior, or evaluation?
- **Likely cause:** what assumption broke?
- **First debug action:** what is the smallest thing to inspect?

## Mini-project and mastery checklist

**Mini-project:** Record a tiny demonstration dataset and train a clone; then intentionally test off-distribution states.

Mastery checklist:

- [ ] I can explain the mental model in one paragraph.
- [ ] I can write the core formula and define every symbol.
- [ ] I can run or read the scratch code and point to the core update/operation.
- [ ] I can name the production framework/API version of the same idea.
- [ ] I can describe where this topic sits in a robot/car/drone/VLA stack.
- [ ] I can name at least three failure modes and one debug action for each.

**Next study steps:** Part 12 imitation learning, Part 10 VLA, Module 16.5 diffusion action heads

## Clone an expert PPO/DQN policy with supervised learning
First train (or load) an expert, then clone it and compare returns.

In [ ]:
from rl.envs import make_env, env_dims
from rl.agents.dqn import DQN, DQNConfig
from rl.agents.bc import BehavioralCloning, BCConfig, collect_demonstrations
from rl.train import train_offpolicy
from rl.utils import ExponentialSchedule, set_seed
import numpy as np

set_seed(0)
env = make_env('LunarLander-v3', seed=0)
s_dim, a_dim, _ = env_dims(env)

# 1) Train a quick expert (use more steps for a stronger teacher)
expert = DQN(s_dim, a_dim, DQNConfig(), device=DEVICE)
train_offpolicy(expert, env, n_steps=120_000,
                eps_schedule=ExponentialSchedule(1.0, 0.01, 0.999), solved_at=200)

# 2) Collect demonstrations and clone them (pure supervised learning)
states, actions = collect_demonstrations(env, expert, n_episodes=50)
bc = BehavioralCloning(s_dim, a_dim, discrete=True, cfg=BCConfig(epochs=80), device=DEVICE)
bc.fit(states, actions)

### Evaluate the clone — and witness covariate shift

In [ ]:
import numpy as np
def eval_agent(agent, n=20, greedy_dqn=False):
    rets = []
    for _ in range(n):
        s, _ = env.reset(); done=False; R=0
        while not done:
            a = agent.act(s, eps=0.0) if greedy_dqn else agent.act(s)
            s, r, t, tr, _ = env.step(a); R += r; done = t or tr
        rets.append(R)
    return np.mean(rets)
print('expert return :', round(eval_agent(expert, greedy_dqn=True), 1))
print('clone  return :', round(eval_agent(bc), 1))
print('\nIf the clone underperforms the expert, that gap is covariate shift —',
      'states the expert never visited. Fixes: more/recovery data, DAgger, RL fine-tuning.')